# Advanced Problems: Keyword Arguments and Keyword-Only Parameters

This notebook contains advanced practice problems with full solutions.

Topics covered:

- Positional arguments
- Keyword arguments
- Keyword-only parameters
- `*args`
- Bare `*`
- Defaults for keyword-only parameters
- Function API design
- Debugging `TypeError`s
- Writing safer and clearer function signatures

## Problem 1: Predict the Output

Given the function below, predict the output of each function call before running it.

In [1]:
def report(a, b=10, *args, mode='summary', debug=False):
    print('a =', a)
    print('b =', b)
    print('args =', args)
    print('mode =', mode)
    print('debug =', debug)

report(1)
print('---')
report(1, 2, 3, 4, mode='full')
print('---')
report(1, debug=True)
print('---')
report(1, 2, mode='compact', debug=True)

a = 1
b = 10
args = ()
mode = summary
debug = False
---
a = 1
b = 2
args = (3, 4)
mode = full
debug = False
---
a = 1
b = 10
args = ()
mode = summary
debug = True
---
a = 1
b = 2
args = ()
mode = compact
debug = True


### Solution 1

`a` is required positional-or-keyword.

`b` is optional positional-or-keyword.

`*args` collects extra positional arguments.

`mode` and `debug` are keyword-only because they appear after `*args`.

In [2]:
# Expected output:
# a = 1
# b = 10
# args = ()
# mode = summary
# debug = False
# ---
# a = 1
# b = 2
# args = (3, 4)
# mode = full
# debug = False
# ---
# a = 1
# b = 10
# args = ()
# mode = summary
# debug = True
# ---
# a = 1
# b = 2
# args = ()
# mode = compact
# debug = True

## Problem 2: Fix the Broken Calls

The following function requires `sep` and `end` to be passed by keyword only.

Fix each invalid call.

In [3]:
def join_values(*values, sep, end=''):
    return sep.join(str(value) for value in values) + end

# Broken calls:
# join_values(1, 2, 3, '-')
# join_values(1, 2, 3)
# join_values(1, 2, 3, '-', '\n')

### Solution 2

In [4]:
print(join_values(1, 2, 3, sep='-'))
print(join_values(1, 2, 3, sep=','))
print(join_values(1, 2, 3, sep='-', end='\n'))

1-2-3
1,2,3
1-2-3



The key idea is that after `*values`, all remaining parameters must be supplied using keyword arguments.

## Problem 3: Design a Safer Function Signature

You are writing a function that sends a notification.

Requirements:

- `user_id` should be required.
- `message` should be required.
- `urgent`, `retries`, and `dry_run` should be keyword-only.
- `urgent` should default to `False`.
- `retries` should default to `3`.
- `dry_run` should default to `False`.

Write the function signature and a small implementation that returns a dictionary.

### Solution 3

In [5]:
def send_notification(user_id, message, *, urgent=False, retries=3, dry_run=False):
    return {
        'user_id': user_id,
        'message': message,
        'urgent': urgent,
        'retries': retries,
        'dry_run': dry_run
    }

print(send_notification(42, 'Server restarted'))
print(send_notification(42, 'Disk almost full', urgent=True, retries=5))
print(send_notification(42, 'Test message', dry_run=True))

{'user_id': 42, 'message': 'Server restarted', 'urgent': False, 'retries': 3, 'dry_run': False}
{'user_id': 42, 'message': 'Disk almost full', 'urgent': True, 'retries': 5, 'dry_run': False}
{'user_id': 42, 'message': 'Test message', 'urgent': False, 'retries': 3, 'dry_run': True}


Using a bare `*` makes every parameter after it keyword-only. This prevents confusing calls like:

`send_notification(42, 'Disk almost full', True, 5, False)`

That call is less readable and more error-prone.

## Problem 4: Explain the Error

For each call below, explain why it fails.

In [6]:
def configure(host, port=5432, *, timeout, ssl=False):
    return host, port, timeout, ssl

# A
# configure('localhost')

# B
# configure('localhost', 5432, 10)

# C
# configure(host='localhost', port=5432, timeout=10, ssl=True)

# D
# configure('localhost', timeout=10, ssl=True)

### Solution 4

A fails because `timeout` is required and keyword-only.

B fails because `timeout` appears after `*`, so it cannot be passed positionally.

C works because every argument is passed correctly.

D works because `host` is positional, `port` uses its default, and `timeout` and `ssl` are passed by keyword.

In [7]:
print(configure(host='localhost', port=5432, timeout=10, ssl=True))
print(configure('localhost', timeout=10, ssl=True))

('localhost', 5432, 10, True)
('localhost', 5432, 10, True)


## Problem 5: Build a Mini Query Function

Write a function named `query_users`.

Requirements:

- Accept any number of user IDs positionally.
- Require `active` as a keyword-only argument.
- Allow optional keyword-only arguments `limit=10` and `sort_by='name'`.
- Return a dictionary containing all arguments.

### Solution 5

In [8]:
def query_users(*user_ids, active, limit=10, sort_by='name'):
    return {
        'user_ids': user_ids,
        'active': active,
        'limit': limit,
        'sort_by': sort_by
    }

print(query_users(101, 102, 103, active=True))
print(query_users(active=False, limit=5))
print(query_users(201, 202, active=True, limit=20, sort_by='created_at'))

{'user_ids': (101, 102, 103), 'active': True, 'limit': 10, 'sort_by': 'name'}
{'user_ids': (), 'active': False, 'limit': 5, 'sort_by': 'name'}
{'user_ids': (201, 202), 'active': True, 'limit': 20, 'sort_by': 'created_at'}


Because `active` has no default value, callers must provide it explicitly.

## Problem 6: Refactor an Ambiguous API

The function below is hard to read when called with many positional arguments.

Refactor it so that `currency`, `include_tax`, and `discount` must be keyword-only.

In [9]:
def old_invoice_total(amount, currency='USD', include_tax=True, discount=0):
    total = amount - discount
    if include_tax:
        total *= 1.2
    return f'{total:.2f} {currency}'

print(old_invoice_total(100, 'EUR', False, 10))

90.00 EUR


### Solution 6

In [10]:
def invoice_total(amount, *, currency='USD', include_tax=True, discount=0):
    total = amount - discount
    if include_tax:
        total *= 1.2
    return f'{total:.2f} {currency}'

print(invoice_total(100, currency='EUR', include_tax=False, discount=10))
print(invoice_total(100, discount=25))

90.00 EUR
90.00 USD


The new version makes the call site clearer:

`invoice_total(100, currency='EUR', include_tax=False, discount=10)`

This is much easier to understand than:

`old_invoice_total(100, 'EUR', False, 10)`

## Problem 7: Validate Keyword-Only Arguments

Write a function `resize_image`.

Requirements:

- `filename` is required.
- `width` and `height` are required keyword-only arguments.
- `keep_aspect_ratio` is optional keyword-only and defaults to `True`.
- Raise `ValueError` if `width` or `height` is not positive.
- Return a dictionary describing the resize operation.

### Solution 7

In [11]:
def resize_image(filename, *, width, height, keep_aspect_ratio=True):
    if width <= 0:
        raise ValueError('width must be positive')
    if height <= 0:
        raise ValueError('height must be positive')

    return {
        'filename': filename,
        'width': width,
        'height': height,
        'keep_aspect_ratio': keep_aspect_ratio
    }

print(resize_image('photo.png', width=800, height=600))
print(resize_image('banner.png', width=1200, height=300, keep_aspect_ratio=False))

{'filename': 'photo.png', 'width': 800, 'height': 600, 'keep_aspect_ratio': True}
{'filename': 'banner.png', 'width': 1200, 'height': 300, 'keep_aspect_ratio': False}


This signature is safer because `width` and `height` cannot accidentally be confused with other positional values.

## Problem 8: Detect Which Calls Are Valid

For the following function, decide whether each call is valid or invalid.

In [12]:
def pipeline(source, *steps, strict=True, retries, verbose=False):
    return {
        'source': source,
        'steps': steps,
        'strict': strict,
        'retries': retries,
        'verbose': verbose
    }

# A: pipeline('data.csv', retries=3)
# B: pipeline('data.csv', 'clean', 'normalize', retries=3)
# C: pipeline('data.csv', 'clean', False, 3)
# D: pipeline(source='data.csv', retries=3, verbose=True)
# E: pipeline('data.csv', strict=False)

### Solution 8

A is valid.

B is valid.

C is invalid because `strict` and `retries` are keyword-only. The values `False` and `3` are collected into `steps` instead.

D is valid.

E is invalid because `retries` is required and has no default.

In [13]:
print(pipeline('data.csv', retries=3))
print(pipeline('data.csv', 'clean', 'normalize', retries=3))
print(pipeline(source='data.csv', retries=3, verbose=True))

{'source': 'data.csv', 'steps': (), 'strict': True, 'retries': 3, 'verbose': False}
{'source': 'data.csv', 'steps': ('clean', 'normalize'), 'strict': True, 'retries': 3, 'verbose': False}
{'source': 'data.csv', 'steps': (), 'strict': True, 'retries': 3, 'verbose': True}


## Problem 9: Build a Strict Configuration Function

Write a function `create_config` that accepts no positional arguments.

Requirements:

- `app_name` is required.
- `debug` defaults to `False`.
- `database_url` is required.
- `cache_enabled` defaults to `True`.
- Return a dictionary.

All parameters must be keyword-only.

### Solution 9

In [14]:
def create_config(*, app_name, debug=False, database_url, cache_enabled=True):
    return {
        'app_name': app_name,
        'debug': debug,
        'database_url': database_url,
        'cache_enabled': cache_enabled
    }

config = create_config(
    app_name='InventoryService',
    database_url='postgresql://localhost/inventory',
    debug=True
)

print(config)

{'app_name': 'InventoryService', 'debug': True, 'database_url': 'postgresql://localhost/inventory', 'cache_enabled': True}


The bare `*` means this function accepts no positional arguments at all.

## Problem 10: Advanced Debugging with `inspect.signature`

Python can inspect function signatures at runtime.

Use `inspect.signature` to determine which parameters are positional, variadic positional, and keyword-only.

### Solution 10

In [15]:
import inspect

def analyze(a, b=0, *args, c, d=10, **kwargs):
    pass

sig = inspect.signature(analyze)

for name, param in sig.parameters.items():
    print(name, '->', param.kind, ', default =', param.default)

a -> POSITIONAL_OR_KEYWORD , default = <class 'inspect._empty'>
b -> POSITIONAL_OR_KEYWORD , default = 0
args -> VAR_POSITIONAL , default = <class 'inspect._empty'>
c -> KEYWORD_ONLY , default = <class 'inspect._empty'>
d -> KEYWORD_ONLY , default = 10
kwargs -> VAR_KEYWORD , default = <class 'inspect._empty'>


Expected parameter kinds:

- `a`: positional-or-keyword
- `b`: positional-or-keyword
- `args`: variadic positional
- `c`: keyword-only
- `d`: keyword-only
- `kwargs`: variadic keyword

## Problem 11: Implement a Decorator-Friendly Function

Write a function `log_event` that accepts:

- Any number of positional event parts.
- Required keyword-only `level`.
- Optional keyword-only `timestamp=None`.
- Optional keyword-only `metadata=None`.

Return a dictionary. Avoid using a mutable object as the default for `metadata`.

### Solution 11

In [16]:
from datetime import datetime

def log_event(*parts, level, timestamp=None, metadata=None):
    if timestamp is None:
        timestamp = datetime.now().isoformat(timespec='seconds')
    if metadata is None:
        metadata = {}

    return {
        'message': ' '.join(str(part) for part in parts),
        'level': level,
        'timestamp': timestamp,
        'metadata': metadata
    }

print(log_event('User', 42, 'logged in', level='INFO'))
print(log_event('Payment failed', level='ERROR', metadata={'order_id': 'A100'}))

{'message': 'User 42 logged in', 'level': 'INFO', 'timestamp': '2026-05-08T16:48:36', 'metadata': {}}
{'message': 'Payment failed', 'level': 'ERROR', 'timestamp': '2026-05-08T16:48:36', 'metadata': {'order_id': 'A100'}}


`metadata=None` is safer than `metadata={}` because default argument values are created once when the function is defined.

## Problem 12: Final Challenge — Build a Flexible Report Generator

Write a function `generate_report`.

Requirements:

- Accept a required `title`.
- Accept any number of positional `sections`.
- Require keyword-only `author`.
- Optional keyword-only `include_toc=False`.
- Optional keyword-only `uppercase_title=False`.
- Optional keyword-only `max_sections=None`.
- If `max_sections` is not `None`, include only that many sections.
- Return a formatted string.

### Solution 12

In [17]:
def generate_report(
    title,
    *sections,
    author,
    include_toc=False,
    uppercase_title=False,
    max_sections=None
):
    if uppercase_title:
        title = title.upper()

    if max_sections is not None:
        if max_sections < 0:
            raise ValueError('max_sections cannot be negative')
        sections = sections[:max_sections]

    lines = []
    lines.append(title)
    lines.append(f'Author: {author}')
    lines.append('')

    if include_toc:
        lines.append('Table of Contents')
        for index, section in enumerate(sections, start=1):
            lines.append(f'{index}. {section}')
        lines.append('')

    for index, section in enumerate(sections, start=1):
        lines.append(f'Section {index}: {section}')

    return '\n'.join(lines)

report = generate_report(
    'Quarterly Results',
    'Revenue',
    'Expenses',
    'Forecast',
    author='Dana',
    include_toc=True,
    uppercase_title=True,
    max_sections=2
)

print(report)

QUARTERLY RESULTS
Author: Dana

Table of Contents
1. Revenue
2. Expenses

Section 1: Revenue
Section 2: Expenses


## Best Practices Summary

- Use keyword-only parameters for options, flags, and configuration.
- Use a bare `*` when you want to forbid positional arguments after a certain point.
- Use `*args` when you want to accept an arbitrary number of positional arguments.
- Required keyword-only parameters have no default value.
- Optional keyword-only parameters have default values.
- Keyword-only parameters improve readability at the call site.
- Prefer `None` as a default when the real default should be a new mutable object.
- Design function signatures to make incorrect usage difficult.